# Build the Agent Loop with the OpenAI Agents SDK

An **agent** is an LLM that can use tools and keep working until it has enough information to finish the task.

The core logic is the same as the raw API version: the model decides what it needs, Python controls which tools are actually allowed to run, tool results go back to the model, and the process continues until there is a final answer.

The difference is **where the loop mechanics live**.

When you code the loop yourself, you have to manage every protocol detail: tool schemas, model responses, function-call IDs, tool outputs, conversation history, retries, and stopping conditions. That is useful when you are learning how agents work, because you can see every moving part.

When you use the **OpenAI Agents SDK**, you keep the same agent logic but move the repetitive plumbing into `Runner.run(...)`. The SDK gives you a cleaner place to define the agent, attach Python tools with `@function_tool`, inspect results, stream output, add tracing, and later extend the workflow with sessions, guardrails, or handoffs.

So this notebook is not changing the idea of the agent loop. It is changing the level of abstraction: from hand-assembled API messages to an SDK runtime designed for agent workflows.

## Today's Goal

Build one small agent that can check where the International Space Station is right now and turn that live data into a mission-control style update.

The task is finished when the SDK returns a result with `final_output`.

In [19]:
import os
from pprint import pprint

import requests

# If you haven't already, install the following packages:
# !pip install python-dotenv openai-agents
from dotenv import load_dotenv
from agents import Agent, Runner, function_tool

TIMEOUT = 60

## Load the API Key

The real key lives in `.env`. The Agents SDK reads `OPENAI_API_KEY` from the environment.

In [20]:
load_dotenv()

os.getenv("OPENAI_API_KEY")
MODEL = "gpt-4.1-mini"

print("OPENAI_API_KEY loaded")
print("Model:", MODEL)

OPENAI_API_KEY loaded
Model: gpt-4.1-mini


## Step 1: Write Normal Python Logic

Start with normal Python functions. This one calls two public APIs: one to track the International Space Station, and one to turn its latitude and longitude into a readable place name.

No extra API key is needed for these public APIs.

In [21]:
def get_place_name(latitude, longitude):
    reverse_geocode_url = "https://api.bigdatacloud.net/data/reverse-geocode-client"
    params = {
        "latitude": latitude,
        "longitude": longitude,
        "localityLanguage": "en",
    }
    response = requests.get(reverse_geocode_url, params=params, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    place_parts = [
        data.get("locality") or data.get("city"),
        data.get("principalSubdivision"),
        data.get("countryName"),
    ]
    place_name = ", ".join(part for part in place_parts if part)

    if place_name:
        return place_name

    return "an area with no nearby place name"


def fetch_iss_location():
    iss_url = "https://api.wheretheiss.at/v1/satellites/25544"
    response = requests.get(iss_url, timeout=TIMEOUT)
    response.raise_for_status()
    data = response.json()

    latitude = data["latitude"]
    longitude = data["longitude"]

    return {
        "place_name": get_place_name(latitude, longitude),
        "latitude": round(latitude, 2),
        "longitude": round(longitude, 2),
        "altitude_km": round(data["altitude"], 2),
        "velocity_km_per_hour": round(data["velocity"], 2),
    }

In [22]:
pprint(fetch_iss_location())

{'altitude_km': 426.45,
 'latitude': 50.27,
 'longitude': 103.57,
 'place_name': 'Zakamensk, Respublika Buryatiya, Russian Federation (the)',
 'velocity_km_per_hour': 27578.38}


## Step 2: Turn the Python Logic into an Agent Tool

The `@function_tool` decorator tells the Agents SDK that the agent is allowed to call this function.

The SDK uses the function name, type hints, and docstring to describe the tool to the model.

In [23]:
@function_tool
def get_iss_location() -> dict:
    """Get the current place name, latitude, longitude, altitude, and speed of the International Space Station."""
    return fetch_iss_location()

## Step 3: Define the Agent

An `Agent` bundles together the model, instructions, and tools.

This is where the raw API version's separate `tools` list becomes part of the agent definition.

In [24]:
iss_agent = Agent(
    name="ISS Mission Control",
    instructions=(
        "You are a concise mission-control communicator. "
        "For live ISS location questions, use get_iss_location before answering. "
        "Report the current place name, coordinates, altitude, and speed. "
        "Make the update exciting but factual."
    ),
    model=MODEL,
    tools=[get_iss_location],
)

## Step 4: Run the Agent

`Runner.run(...)` is the agent loop.

During this one run, the SDK can call the model, notice that the model requested `get_iss_location`, run the Python tool, send the tool result back to the model, and return the final answer.

In [25]:
task = (
    "Where is the International Space Station right now? "
)

result = await Runner.run(iss_agent, task)

print(result.final_output)

The International Space Station is currently flying over Kharatsay, Respublika Buryatiya, Russian Federation. It's at an altitude of approximately 426.46 kilometers, orbiting Earth at an incredible speed of about 27,578.47 kilometers per hour!


## Optional: Inspect the Result

For most beginner projects, `result.final_output` is the main thing you need.

The result also tells you which agent finished the run. In larger projects, that matters when handoffs move control between specialists.

In [26]:
print("Last agent:", result.last_agent.name)
print("Final output:")
print(result.final_output)

Last agent: ISS Mission Control
Final output:
The International Space Station is currently flying over Kharatsay, Respublika Buryatiya, Russian Federation. It's at an altitude of approximately 426.46 kilometers, orbiting Earth at an incredible speed of about 27,578.47 kilometers per hour!


## What Changed from the Raw API Version

- `@function_tool` replaces the hand-written function schema.
- `Agent(...)` replaces the separate model, instructions, and tools request body.
- `Runner.run(...)` replaces the manual `while True` loop.
- `result.final_output` replaces digging through response JSON for the final text.
- Your local Python still controls the actual tool implementation.

## What Happened

Python and the SDK each had a job:

- You defined an agent and gave it one Python tool.
- `Runner.run(...)` started the agent loop.
- The model decided it needed current ISS data.
- The SDK called your `get_iss_location` tool.
- Your Python code called the public ISS APIs.
- The SDK sent the tool result back to the model.
- The model wrote the final mission update.

## Quick Review

- A **tool** is a Python function the agent is allowed to call.
- `@function_tool` exposes local Python logic to the agent.
- An **agent** combines instructions, a model, and tools.
- `Runner.run(...)` runs the agent loop until there is a final answer.
- `result.final_output` is the final answer you show the user.